# Supplementary Table 3

PREDICT antibiotic endpoints (human-assigned share of total LFQ intensity and quantified protein groups) under four definitions of antibiotic exposure that differ only in how rifaximin recorded for hepatic encephalopathy is handled. The first section rebuilds the published workbook from `source_data/STab3/` and checks it cell by cell. The second recomputes every statistic and count from the controlled PREDICT inputs when `FASTALAKE_CONTROLLED_DIR` is set, and otherwise skips.

Run from the repository root: `python paper.py build STab3`.

In [1]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered outputs; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()

## Table

Rebuilds `Supplementary_Table_3.xlsx` (sheets Read_me, Endpoints, Composition) from `endpoints_by_definition.tsv` and `composition_by_definition.tsv` with the code that assembled the published workbook, then compares every cell value with `expected/Supplementary_Table_3.xlsx`. Receipt: `checks/workbook.json`.

In [2]:
import json
import pandas as pd
built = panels.build_workbook(SD, OUT / "Supplementary_Table_3.xlsx")
receipt = panels.compare_workbooks(built, UNIT / "expected" / "Supplementary_Table_3.xlsx")
receipt.update(built="build/STab3/Supplementary_Table_3.xlsx", expected="tables/STab3/expected/Supplementary_Table_3.xlsx")
(UNIT / "checks").mkdir(exist_ok=True)
(UNIT / "checks" / "workbook.json").write_text(json.dumps(receipt, indent=1) + "\n")
print(receipt["status"], receipt["cells_compared"], "cells,", receipt["n_differences"], "differences")
pd.set_option("display.width", 250, "display.max_columns", 30)
panels.read_endpoints(SD)[["scenario", "metric", "n_patients", "mean_on_minus_off", "CI95_low", "CI95_high",
                          "p_signflip_MC", "q_BH_two_endpoints_within_definition"]]

IDENTICAL 213 cells, 0 differences


,scenario,metric,n_patients,mean_on_minus_off,CI95_low,CI95_high,p_signflip_MC,q_BH_two_endpoints_within_definition
0,A_current,Human-assigned share of total LFQ intensity,42,11.133322,5.559830,17.156839,0.000340,0.000680
1,A_current,Quantified protein groups,42,-2213.952381,-3469.322024,-997.525595,0.001255,0.001255
2,B_HE_as_exposure,Human-assigned share of total LFQ intensity,40,11.469654,5.745656,17.654898,0.000240,0.000480
3,B_HE_as_exposure,Quantified protein groups,40,-2378.062500,-3651.466250,-1140.035000,0.000540,0.000540
4,C_HE_patients_excluded,Human-assigned share of total LFQ intensity,35,12.302762,5.821377,19.302561,0.000710,0.000710
5,C_HE_patients_excluded,Quantified protein groups,35,-2665.714286,-4071.364286,-1307.596429,0.000605,0.000710
6,D_HE_off_rows_excluded,Human-assigned share of total LFQ intensity,38,11.744018,5.733085,18.146941,0.000375,0.000750
7,D_HE_off_rows_excluded,Quantified protein groups,38,-2382.842105,-3716.698355,-1093.060855,0.001305,0.001305


## Statistics (controlled inputs)

Supports Results, v28d line 137: "Counting rifaximin given for hepatic encephalopathy as antibiotic exposure left 40 paired patients, with an 11.5 percentage-point higher human-assigned share (95% CI 5.7–17.7, q=0.00048) and 2,378 fewer protein groups (95% CI 1,140–3,651, q=0.00054, Supplementary Table 3)", and the counts in Methods, v28d line 793. Unit: patient; n = 42, 40, 35 and 38 paired patients (definitions A to D) from 306 acquisitions. Test: mean paired on-minus-off difference, percentile patient bootstrap (50,000 draws), two-sided Monte Carlo sign-flip test (200,000 draws, plus-one correction), Benjamini–Hochberg across the two endpoints within each definition. A planted negative control (every hepatic-encephalopathy flag inverted) must leave definition A unchanged and change definition B.

Needs `PREDICT_metadata_2026-08-03.tsv` and `acquisition_accounting.tsv` in `FASTALAKE_CONTROLLED_DIR` (see `controlled/CONTROLLED_INPUTS.md`). Only aggregate values are printed.

In [3]:
primary = pd.read_csv(SD / "primary_paired_statistics.tsv", sep="\t")
loaded = panels.load_controlled()
res = None
if loaded is not None:
    d, ac = loaded
    res = panels.stat_definitions(d, ac, primary)
    cmp = panels.compare_to_source(res, SD)
    print(json.dumps(cmp, indent=1))
    assert cmp["stats_identical_to_producer_output"] and cmp["composition_identical"] and cmp["counts_identical"]
    neg = panels.stat_definitions(d, ac, primary, negative_control=True)
    print("negative control: A n =", neg["counts"]["paired_A_current"]["n"], "| B n =", neg["counts"]["paired_B_HE_as_exposure"]["n"])
    assert neg["counts"]["paired_A_current"]["n"] == 42 and neg["counts"]["paired_B_HE_as_exposure"]["n"] != 40
    del d, ac
    display(res["stats"][["scenario", "metric", "n_patients", "mean_on_minus_off", "CI95_low", "CI95_high",
                          "signflip_extreme_count", "p_signflip_MC", "q_two_clinical_endpoints", "acq_off", "acq_on"]])

{
 "stats_identical_to_producer_output": true,
 "stats_max_rel_diff_vs_published_table": 4.166293554593504e-13,
 "composition_identical": true,
 "counts_identical": true
}


negative control: A n = 42 | B n = 7


,scenario,metric,n_patients,mean_on_minus_off,CI95_low,CI95_high,signflip_extreme_count,p_signflip_MC,q_two_clinical_endpoints,acq_off,acq_on
0,A_current,Human,42,11.133322,5.559830,17.156839,67,0.000340,0.000680,67,88
1,A_current,pg_parsimony,42,-2213.952381,-3469.322024,-997.525595,250,0.001255,0.001255,67,88
2,B_HE_as_exposure,Human,40,11.469654,5.745656,17.654898,47,0.000240,0.000480,63,87
3,B_HE_as_exposure,pg_parsimony,40,-2378.062500,-3651.466250,-1140.035000,107,0.000540,0.000540,63,87
4,C_HE_patients_excluded,Human,35,12.302762,5.821377,19.302561,141,0.000710,0.000710,53,75
5,C_HE_patients_excluded,pg_parsimony,35,-2665.714286,-4071.364286,-1307.596429,120,0.000605,0.000710,53,75
6,D_HE_off_rows_excluded,Human,38,11.744018,5.733085,18.146941,74,0.000375,0.000750,58,82
7,D_HE_off_rows_excluded,pg_parsimony,38,-2382.842105,-3716.698355,-1093.060855,260,0.001305,0.001305,58,82


## Checks and numbers

The table check runs above (`checks/workbook.json`); this unit has no figure panels, so there are no pixel checks. The numbers below are printed in `NUMBERS.tsv` order, at full precision, from the controlled recomputation.

In [4]:
print("workbook check:", receipt["status"])
if res is None:
    print("numbers SKIPPED: controlled inputs absent")
else:
    for label, value in panels.numbers(res):
        print(f"{label}\t{value!r}")

workbook check: IDENTICAL
137 B paired patients	40
137 B human share mean on-off (pp)	11.469653688156376
137 B human share CI low	5.745655758818914
137 B human share CI high	17.654897529445666
137 B human share q	0.00047999760001199996
137 B depth fewer groups (-mean on-off)	2378.0625
137 B depth CI, smaller reduction (-CI high)	1140.0349999999999
137 B depth CI, larger reduction (-CI low)	3651.46625
137 B depth q	0.0005399973000134999
479 A patients	42
479 B patients	40
479 C patients	35
479 D patients	38
793 Off acquisitions reclassified A->B	12
793 patients of those acquisitions	'seven'
793 B paired patients	40
793 C patients	35
793 A patients	42
793 Off acquisitions with HE record set Unknown (D)	12
793 D patients	38
893 endpoints per BH family	'two'
